# 7 — Memory (Agent Engine Sessions + Memory Bank)

**AWS-side equivalent:** AgentCore Memory — `notebooks/aws/07-agentcore-memory.ipynb` (semantic / user-preference / summary strategies, session add+search). 
**GCP service:** Vertex AI [Agent Engine Sessions](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/sessions) (event log per `(user_id, session_id)`) + [Memory Bank](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/overview) (LLM-extracted long-term memories scoped by user).

Sessions hold the raw event log; Memory Bank extracts and self-curates long-term facts from that event log. They're attached to an `agent_engine` (formerly *reasoning engine*) resource. 

Vertex AI **Reasoning Engine** is the older name; the current resource name on the API side is still `reasoningEngines` but every doc and SDK calls it **Agent Engine**. Same object, the rename is purely cosmetic.

## Service surface (as of 2026-05-20)

- Client: `vertexai.Client(project=..., location=...)` → `client.agent_engines.*`
- Resource API path: `projects/{p}/locations/{l}/reasoningEngines/{id}`
- Sessions:  `client.agent_engines.sessions.create / get / list / events.append`
- Memories:  `client.agent_engines.memories.generate / retrieve / list / delete`
- Scope is opaque key/value (commonly `{"user_id": ...}`) — memories generated from a session inherit `{"user_id": <session.user_id>}` unless overridden.

Docs: <https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/memory-bank/quickstart-api>

## Step 1: Config + client

In [ ]:
import json
import datetime as dt
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

GEMINI_MODEL = gcp.get('text_model', 'gemini-2.5-flash')
tracker = ResourceTracker()

import vertexai
client = vertexai.Client(project=PROJECT_ID, location=REGION)
print(f'✓ vertexai.Client bound to project={PROJECT_ID} location={REGION}')

## Step 2: Provision (or reuse) an Agent Engine instance

Memory Bank lives on an Agent Engine resource; we provision a minimal one (no deployed agent code — just a host for sessions + memories). If `gcp.agent_engine_id` is already set from a prior run we reuse it.

In [ ]:
from google import genai as _genai  # only for type imports in callers

AGENT_ENGINE_ID = gcp.get('agent_engine_id')

if AGENT_ENGINE_ID:
    agent_engine = client.agent_engines.get(
        name=f'projects/{PROJECT_ID}/locations/{REGION}/reasoningEngines/{AGENT_ENGINE_ID}'
    )
    print(f'✓ Reusing Agent Engine: {agent_engine.api_resource.name}')
else:
    agent_engine = client.agent_engines.create(
        config={
            'display_name': 'orgiq-memory-host',
            'description': 'OrgIQ Agent Engine instance hosting Memory Bank for org assistants.',
            # Default memory-bank config: extraction enabled, scope keyed by user_id.
            'context_spec': {
                'memory_bank_config': {
                    'generation_config': {'model': f'publishers/google/models/{GEMINI_MODEL}'},
                }
            },
        }
    )
    AGENT_ENGINE_ID = agent_engine.api_resource.name.split('/')[-1]
    gcp['agent_engine_id'] = AGENT_ENGINE_ID
    Path('config.json').write_text(json.dumps(config, indent=2))
    tracker.track('agent_engine', AGENT_ENGINE_ID, region=REGION)
    tracker.flush()
    print(f'✓ Created Agent Engine: {agent_engine.api_resource.name}')

ENGINE_NAME = agent_engine.api_resource.name
print(f'  resource = {ENGINE_NAME}')

## Step 3: Create a session for a user

In [ ]:
USER_ID = 'user-alice-chen'

session = client.agent_engines.sessions.create(
    name=ENGINE_NAME,
    user_id=USER_ID,
)
SESSION_NAME = session.response.name  # full resource path
print(f'✓ session created: {SESSION_NAME}')
print(f'  user_id = {USER_ID}')

## Step 4: Append a multi-turn conversation as session events

Mirrors the 6-message `ConversationalMessage` script in the AWS notebook.

In [ ]:
conversation = [
    ('user',      'Who are the best machine learning experts in our org?'),
    ('assistant', 'Bob Kumar is our top ML expert — Staff ML Engineer, strength 0.95, author of the ML Pipeline Architecture doc.'),
    ('user',      'I prefer results sorted by recency, not score. I am on the platform team.'),
    ('assistant', 'Noted — I will prioritize recent contributions and weigh platform-team relevance for you.'),
    ('user',      'Find docs on CI/CD pipelines.'),
    ('assistant', 'Dave Park\'s "GitOps Pipeline Setup" is the top hit — Dave is on platform with CI/CD strength 0.93.'),
]

for i, (author, text) in enumerate(conversation):
    client.agent_engines.sessions.events.append(
        name=SESSION_NAME,
        author=author,
        invocation_id=f'turn-{i:02d}',
        timestamp=dt.datetime.now(tz=dt.timezone.utc),
        config={'content': {'role': author, 'parts': [{'text': text}]}},
    )

print(f'✓ {len(conversation)} events appended to {SESSION_NAME.split("/")[-1]}')

## Step 5: Generate long-term memories from this session

Memory Bank runs an LLM extraction over the session's events and stores facts under the session's user scope.

In [ ]:
gen = client.agent_engines.memories.generate(
    name=ENGINE_NAME,
    vertex_session_source={'session': SESSION_NAME},
    scope={'user_id': USER_ID},
)
print('✓ memory generation kicked off')
print(f'  operation/result: {gen}')

## Step 6: Retrieve memories for a new turn

On the next user message, the agent calls `memories.retrieve` keyed by the user scope to pull facts that should bias the LLM's response.

In [ ]:
memories = list(
    client.agent_engines.memories.retrieve(
        name=ENGINE_NAME,
        scope={'user_id': USER_ID},
    )
)
print(f'✓ retrieved {len(memories)} memories for {USER_ID}')
for m in memories[:10]:
    # Each returned item carries a `memory` payload — the schema is still being stabilized; print loosely.
    fact = getattr(getattr(m, 'memory', m), 'fact', None) or str(m)
    print(f'  - {fact[:200]}')

## Step 7: Use retrieved memories to ground a fresh Gemini turn

This is the practical "context augmentation on the next user turn" pattern — same shape as AWS step 10 (search_memories then format into the next prompt).

In [ ]:
from google import genai
from google.genai import types

gen_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

memory_block = '\n'.join(
    f'- {getattr(getattr(m, "memory", m), "fact", str(m))}' for m in memories[:8]
) or '(no prior memories yet — Memory Bank may still be extracting)'

next_user_turn = 'Help me pick someone to pair with on a new CI/CD project.'

system_with_memory = (
    'You are OrgIQ Search. Use the user-specific memories below to bias your suggestion. '
    'Cite people by name.\n\n'
    f'KNOWN ABOUT THIS USER:\n{memory_block}'
)

resp = gen_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=next_user_turn,
    config=types.GenerateContentConfig(
        system_instruction=system_with_memory,
        temperature=0.3,
        max_output_tokens=512,
    ),
)
print('User:', next_user_turn)
print('\nAssistant (memory-grounded):\n', resp.text)

## Step 8: Second session — verify cross-session persistence

Memories are scoped to the user, not the session, so a fresh session should see the same facts.

In [ ]:
session_b = client.agent_engines.sessions.create(name=ENGINE_NAME, user_id=USER_ID)
print(f'new session: {session_b.response.name}')

memories_b = list(client.agent_engines.memories.retrieve(name=ENGINE_NAME, scope={'user_id': USER_ID}))
print(f'✓ session-2 sees {len(memories_b)} memories (cross-session persistence confirmed)')

## Step 9: Save state

In [ ]:
gcp['agent_engine_id'] = AGENT_ENGINE_ID
gcp['memory_user_scope'] = {'user_id': USER_ID}
Path('config.json').write_text(json.dumps(config, indent=2))
print(f'✓ config.json updated: gcp.agent_engine_id={AGENT_ENGINE_ID}')

## Cross-reference vs AWS

| AWS step (`07-agentcore-memory.ipynb`)         | GCP equivalent here                                |
|------------------------------------------------|----------------------------------------------------|
| `create_memory_store(... strategies=[...])`    | `agent_engines.create(... memory_bank_config=...)` |
| `MemorySessionManager.create_session`          | `agent_engines.sessions.create`                    |
| `add_messages(session_id, messages=[...])`     | `sessions.events.append(name=..., author=..., config={'content': {...}})` |
| `extract_memories(...)`                        | `memories.generate(vertex_session_source=...)`     |
| `search_memories(query=..., user_id=...)`      | `memories.retrieve(scope={'user_id': ...})`        |
| 3 strategies (SEMANTIC, USER_PREFERENCE, SUMMARY) | one extractor today; topic/category filtering tunes the surface |

**Uncertainty:** the `memory.fact` field name surfaced above is best-effort — the public API for the returned memory item is still in flux. If you hit `AttributeError`, dump the raw `m.__dict__` to find the current field name.

**Next:** [08-agent-engine-tools.ipynb](08-agent-engine-tools.ipynb) — register Python tools (and optionally an MCP server) on an Agent Engine.